# Pipeline stages — calling them individually

Every stage in PaddockTS is a plain Python function. `get_outputs(query)` is just an orchestrator that calls them in order on two parallel threads with a live dashboard. For debugging, fine-tuning, or swapping in your own implementation of any step, you can call each stage directly.

Caching is built in — every stage writes a `_SUCCESS` marker after its data file completes, and re-running with the same `Query` is a no-op (or a fast load from disk).

This notebook walks through the Sentinel-2 → PaddockTS chain stage by stage and shows how to inspect each intermediate output.

In [ ]:
from datetime import date
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr

from PaddockTS.query import Query

query = Query(
    bbox=[148.36265, -33.52606, 148.38265, -33.50606],
    start=date(2024, 1, 1),
    end=date(2024, 12, 31),    # full year so phenology metrics are meaningful
    stub="stages_demo",
)
print(query)

## 1. Download Sentinel-2 (`download_sentinel2`)

Searches the Digital Earth Australia STAC catalog (`ga_s2am_ard_3` / `ga_s2bm_ard_3`), loads the requested bands via `odc.stac` on a Dask cluster, and **streams chunks directly to a Zarr store** at `query.sentinel2_path`. Peak memory is `~workers × chunk_size`, independent of AOI size.

Returns an `xarray.Dataset` on `(time, y, x)` with the raw cube — DN values in `[0, 10000]`, plus the `oa_fmask` quality band.

In [ ]:
from PaddockTS.Sentinel2.download_sentinel2 import download_sentinel2

ds = download_sentinel2(query)
print(f"scenes: {ds.time.size}")
print(f"bands : {list(ds.data_vars)}")
print(f"shape : {dict(ds.sizes)}")

## 2. Clean cloud + bad scenes (`clean_sentinel2`)

Applies an fmask-based clear-sky mask (drops cloud + shadow pixels), removes the fmask band itself, then drops entire scenes whose NaN fraction exceeds `max_nan_fraction`. The cleaned cube is written to `query.sentinel2_clean_path` (also Zarr, marker-gated). All downstream stages read from this.

In [ ]:
from PaddockTS.Sentinel2.clean_sentinel2 import clean_sentinel2

ds_clean = clean_sentinel2(query, ds_sentinel2=ds, max_nan_fraction=0.5)
print(f"scenes before: {ds.time.size}")
print(f"scenes after : {ds_clean.time.size}")
print(f"fmask removed: {'oa_fmask' not in ds_clean.data_vars}")

## 3. Compute spectral indices (`compute_indices`)

Adds NDVI, CFI, NIRv, NDTI, CAI as new `(time, y, x)` variables to the dataset. The augmented cube is cached to `query.indices_path`.

In [ ]:
from PaddockTS.SpectralIndices.indices import compute_indices

ds = compute_indices(query, ds_sentinel2=ds_clean)
added = [v for v in ['NDVI', 'CFI', 'NIRv', 'NDTI', 'CAI'] if v in ds.data_vars]
print("added indices:", added)

# Median NDVI as a map
fig, ax = plt.subplots(figsize=(6, 5))
ds.NDVI.median(dim="time").plot.imshow(ax=ax, cmap="RdYlGn", vmin=-0.2, vmax=0.9)
ax.set_title("Median NDVI over the AOI")
plt.show()

## 4. Fractional cover (`compute_fractional_cover`)

Spectral unmixing of the six SR bands into bare ground (`bg`), green vegetation (`pv`), and non-green vegetation (`npv`) — per pixel, per timestep. Uses a TFLite MLP adapted from [`fractionalcover3`](https://github.com/jrsrp/fractionalcover3).

Visualised as a false-colour RGB: red = bare, green = growing, blue = stubble / dry.

In [ ]:
from PaddockTS.FractionalCover import compute_fractional_cover

fc = compute_fractional_cover(query, ds_sentinel2=ds_clean)
print(fc)

In [ ]:
fc_t = fc.isel(time=0)
total = np.maximum(fc_t.bg + fc_t.pv + fc_t.npv, 1e-6)
rgb = np.stack([
    (fc_t.bg / total).values,
    (fc_t.pv / total).values,
    (fc_t.npv / total).values,
], axis=-1)
rgb = np.clip(np.nan_to_num(rgb), 0, 1)

fig, ax = plt.subplots(figsize=(6, 5))
ax.imshow(rgb)
ax.set_title(f"Fractional cover — {str(fc.time.values[0])[:10]}\n"
             "R=bare, G=green veg, B=non-green veg")
ax.axis('off')
plt.show()

## 5. Segment paddocks with SAM (`get_paddocks`)

Three internal steps:

1. **Presegmentation** — derive a single grayscale image from the Sentinel-2 stack using NDWI Fourier features (emphasises stable field boundaries).
2. **SAM** — feed the presegmented image to Segment Anything (ViT-H backbone; ~2.4 GB checkpoint auto-downloaded on first run).
3. **Filter** — explode, reproject to UTM, compute area + isoperimetric compactness, drop polygons outside `[min_area_ha, max_area_ha]` or below `min_compactness`.

Returns a `GeoDataFrame` with one row per paddock; also written to `query.sam_paddocks_path`.

In [ ]:
from PaddockTS.PaddockSegmentation.get_paddocks import get_paddocks

paddocks = get_paddocks(query, ds_sentinel2=ds_clean)
print(f"{len(paddocks)} paddocks")
paddocks[["paddock", "area_ha", "compactness"]].head()

In [ ]:
# Overlay the polygons on the median NDVI to eyeball segmentation quality
ndvi_med = ds.NDVI.median(dim="time")
extent = [float(ds.x.min()), float(ds.x.max()), float(ds.y.min()), float(ds.y.max())]

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(ndvi_med, cmap="RdYlGn", vmin=-0.1, vmax=0.9, extent=extent, origin="upper")
paddocks.boundary.plot(ax=ax, color="red", linewidth=1)
ax.set_title(f"{len(paddocks)} SAM-segmented paddocks over median NDVI")
ax.axis("off")
plt.show()

## 6. Per-paddock time series (`make_paddock_time_series`)

The pivot from pixel-space to paddock-space. Rasterises paddock polygons onto the S2 grid, then for every band computes the per-paddock NaN-aware median at each timestep. Output is an `xarray.Dataset` on `(paddock, time)` persisted as Zarr.

This is the central data product — downstream stages (yearly split, phenology, plots) all consume it.

In [ ]:
from PaddockTS.Phenology.make_paddock_time_series import make_paddock_time_series

ts = make_paddock_time_series(query, ds_sentinel2=ds, paddocks_filepath=query.sam_paddocks_path)
print(ts)

## 7. Yearly split + phenology

`make_yearly_paddock_time_series` splits the cube by calendar year and attaches a `doy` (day-of-year) coordinate. `estimate_phenology` wraps the vendored [`phenolopy`](https://github.com/lewistrotter/phenolopy) to derive SoS / PoS / EoS and friends per paddock per year.

In [ ]:
from PaddockTS.Phenology.make_yearly_paddock_time_series import make_yearly_paddock_time_series
from PaddockTS.Phenology.estimate_phenology import estimate_phenology

yearly = make_yearly_paddock_time_series(query, ds_paddockTS=ts)
phen = estimate_phenology(query, ds_yearly=yearly, variable="NDVI")

for year, df in phen.items():
    print(f"\n=== {year} — {len(df)} paddocks ===")
    print(df[["paddock", "sos_times", "pos_times", "eos_times",
             "aos_values", "los_values", "num_peaks"]].head())

## 8. Phenology curve for a single paddock

Plotted ourselves; for a multi-paddock × multi-year version see `PaddockTS.Plotting.phenology_plot`.

In [ ]:
year = next(iter(phen))
df_year = phen[year]
ds_year = yearly[year]

paddock_id = ts.paddock.values[0]
row = df_year[df_year["paddock"].astype(str) == str(paddock_id)].iloc[0]

fig, ax = plt.subplots(figsize=(10, 4))
ax.scatter(ds_year.doy, ds_year.NDVI.sel(paddock=paddock_id),
           color="blue", s=20, label="NDVI")
ax.axvline(row.sos_times, color="green", linestyle="--", label="SoS")
ax.axvline(row.pos_times, color="blue",  linestyle="-.", label="PoS")
ax.axvline(row.eos_times, color="red",   linestyle=":",  label="EoS")
ax.set_xlabel("DOY")
ax.set_ylabel("NDVI")
ax.set_title(f"Paddock {paddock_id} — {year}")
ax.legend()
plt.tight_layout()
plt.show()

## Caching contract

Every cached output is guarded by a `_SUCCESS` marker (a sibling file for GeoTIFFs / GeoPackages, an internal file inside the directory for Zarrs). On every stage's entry:

1. Check if the data file AND the `_SUCCESS` marker exist.
2. If yes, load from cache and return immediately.
3. If the data file exists but the marker is missing (e.g. previous run was killed mid-write), wipe the partial cache and re-run cleanly.

Re-run any of the cells above — the second call returns instantly from disk. To force a fresh rebuild, pass `reload=True` to `get_outputs` or delete the relevant path manually.

## See also

- [`01_quickstart.ipynb`](01_quickstart.ipynb) — the all-in-one `get_outputs(query)` flow.
- [`03_custom_paddocks.ipynb`](03_custom_paddocks.ipynb) — skip SAM and use your own paddock boundaries.
- [API reference](https://thestochasticman.github.io/paddock-ts-local/api/) — every public function with examples.